# K-nearest neighbors

University of Michigan EECS 498/598, Winter 2022. Compact companion to the completed course assignment.

**Static validation only; not executed.** No ordinary-notebook outputs or scores are copied here. Run cells in order only when execution is intended; training variants retain the ordinary notebook parameters. Existing course dependencies and data are required; no installation or repository cloning is performed. Results are isolated under `core_results/knn`.

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import importlib

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/drive/MyDrive/umich-dlcv-wi2022')]
PROJECT_ROOT = next((p for p in candidates if (p / 'assignments' / 'a1' / 'knn_core.ipynb').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the canonical project or assignment directory; mount Drive first in Colab.')
ASSIGNMENT_DIR = PROJECT_ROOT / 'assignments' / 'a1'
if 'eecs598' in sys.modules and Path(sys.modules['eecs598'].__file__).resolve().parent != (ASSIGNMENT_DIR / 'eecs598').resolve():
    raise RuntimeError('Restart the kernel before switching assignments: eecs598 differs across assignments.')
os.chdir(ASSIGNMENT_DIR)
sys.path.insert(0, str(ASSIGNMENT_DIR))
sys.modules.setdefault('imp', importlib)
import torch
import matplotlib.pyplot as plt
import eecs598
from eecs598 import reset_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = ASSIGNMENT_DIR / 'core_results' / 'knn'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed(0)
print('Device:', device)


## Data

In [ ]:
DATA_DIR = next((p for p in (ASSIGNMENT_DIR, PROJECT_ROOT / 'assignments' / 'a1') if (p / 'cifar-10-batches-py').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Prepare the course CIFAR-10 cache before running this notebook.')
try:
    os.chdir(DATA_DIR)
    x_train, y_train, x_test, y_test = eecs598.data.cifar10()
finally:
    os.chdir(ASSIGNMENT_DIR)

x_train, y_train, x_test, y_test = [t.to(device) for t in (x_train, y_train, x_test, y_test)]
print(x_train.shape, x_test.shape)


## Distance variants

In [ ]:
from knn_core import compute_distances_two_loops, compute_distances_one_loop, compute_distances_no_loops, KnnClassifier, knn_cross_validate, knn_get_best_k
for distance_fn in (compute_distances_two_loops, compute_distances_one_loop, compute_distances_no_loops):
    distances = distance_fn(x_train[:20], x_test[:10])
    print(distance_fn.__name__, distances.shape)


## Cross-validation

In [ ]:
k_to_accuracies = knn_cross_validate(x_train[:5000], y_train[:5000], num_folds=5)
best_k = knn_get_best_k(k_to_accuracies)
for k, accuracies in sorted(k_to_accuracies.items()):
    plt.scatter([k] * len(accuracies), accuracies)
plt.xlabel('k')
plt.ylabel('Validation accuracy (%)')
plt.savefig(RESULTS_DIR / 'cross_validation.png', bbox_inches='tight')
plt.show()
print('Selected k:', best_k)


## Evaluate all test images

In [ ]:
classifier = KnnClassifier(x_train, y_train)
# Chunk test images to bound the full training-by-test distance matrix.
predictions = torch.cat([classifier.predict(x_test[start:start + 100], k=best_k) for start in range(0, len(x_test), 100)])
accuracy = (predictions == y_test).float().mean().item()
print(f'Test accuracy: {100 * accuracy:.2f}%')
torch.save({'best_k': best_k, 'cross_validation': k_to_accuracies, 'test_accuracy': accuracy}, RESULTS_DIR / 'knn_metrics.pt')
